# Safety-Incident Anomaly Pattern Detection

**INT396 · SIH26165**  
This notebook is part of a reproducible project comparing Isolation Forest and Local Outlier Factor on OSHA Severe Injury Reports.

**Interpretation rule:** model flags are statistical outliers, not verified anomalies, accident predictions, or proof of unsafe conduct.

Select the project's Python environment as the notebook kernel. The raw dataset must be available at `data/raw/January2015toNovember2025.csv`.

# Exploratory Data Analysis

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from safety_anomaly.data.io import load_incidents
from safety_anomaly.data.clean import prepare_records
DATA_PATH=ROOT/'data/raw/January2015toNovember2025.csv'
df=prepare_records(load_incidents(DATA_PATH))
print(df.shape)

## 1. Incident distribution over time

In [ ]:
year_counts=df['EventDate'].dt.year.value_counts().sort_index()
ax=year_counts.plot(kind='bar', figsize=(10,4), title='Reported severe injury records by event year')
ax.set_xlabel('Event year'); ax.set_ylabel('Records'); plt.tight_layout(); plt.show()

## 2. Event type distribution

In [ ]:
top_events=df['EventTitle'].fillna('Unknown').value_counts().head(15).sort_values()
ax=top_events.plot(kind='barh', figsize=(10,7), title='Most frequently reported event categories')
ax.set_xlabel('Records'); plt.tight_layout(); plt.show()

## 3. Industry distribution

In [ ]:
top_industry=df['Primary NAICS'].astype('string').fillna('Unknown').value_counts().head(15).sort_values()
ax=top_industry.plot(kind='barh', figsize=(10,6), title='Most frequent Primary NAICS categories')
ax.set_xlabel('Records'); plt.tight_layout(); plt.show()

## 4. Narrative length

In [ ]:
narrative_length=df['Final Narrative'].fillna('').astype(str).str.split().str.len()
display(narrative_length.describe().to_frame('word_count'))
ax=narrative_length.clip(upper=narrative_length.quantile(.99)).plot(kind='hist', bins=40, figsize=(9,4), title='Incident narrative length (99th percentile clipped for display)')
ax.set_xlabel('Word count'); plt.tight_layout(); plt.show()

## 5. Missingness

In [ ]:
missing=(df.isna().mean()*100).sort_values(ascending=False).head(20).sort_values()
ax=missing.plot(kind='barh', figsize=(10,7), title='Highest missingness fields')
ax.set_xlabel('Missing (%)'); plt.tight_layout(); plt.show()

## Interpretation
These plots describe the composition of the reported dataset. They do not establish injury rates or causal relationships because the dataset does not provide workforce exposure denominators.